In [ ]:
%load_ext autoreload
%autoreload 2

# vv4ml Quickstart Guide
Welcome to the introduction tutorial of vv4ml library.

This tutorial will show you how to:
1. install vv4ml
2. generate a mock dataset and train a model of reference
3. run **convergence**, **numerical error** and **smoothness** tests to assess your model credibility.

## Installation

In [ ]:
!pip install vv4ml

## Environment setup and data generation
To run vv4ml, let's generate some data with `scikit-learn` and train a model of reference (in this example `RandomForestRegressor`).

In [ ]:
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

# 1. Mock dataset generation (3 feature and 1 target)
x_raw, y_raw = make_regression(n_samples=2000, n_features=3, noise=0.1, random_state=42)

# Shift data away from zero to stabilize the error computation
y_raw = y_raw + 1000

# Target scaling to mean=0 and variance=1 (because of "+ 1000" shift)
y_train_scaled = MinMaxScaler(feature_range=(1, 2)).fit_transform(y_raw.reshape(-1, 1)).flatten()

# Pandas structure conversion (necessary to be compatible with vv4ml)
x_train = pd.DataFrame(x_raw[:800], columns=['feature_1', 'feature_2', 'feature_3'])
y_train_series = pd.Series(y_train_scaled[:800])
y_train_df = pd.DataFrame(y_train_scaled, columns=['target']) # This is necessary for the numerical error test

x_eval = pd.DataFrame(x_raw[800:], columns=['feature_1', 'feature_2', 'feature_3'])
y_eval = y_train_scaled[800:]

# 2. Model of reference training
model_type = "RandomForestRegressor"
best_params = {"n_estimators": 150, "max_depth": 5, "min_samples_leaf": 8, "random_state": 42}

ref_model = RandomForestRegressor(**best_params)
ref_model.fit(x_train, y_train_series)

# Reference predictions extraction
ref_predictions = ref_model.predict(x_eval)

print(f"Train dataset: {x_train.shape}")
print(f"Evaluation dataset: {x_eval.shape}")

## Convergence test

In [ ]:
from vv4ml.verification.convergence import run_convergence_test

# Test configuration
n_rows_to_test = [50, 100, 200, 300]
threshold_convergence = 0.02 # This is the 2% of the mean relative error tolerance related to the complete model

# Run
convergence_result = run_convergence_test(
    x_train=x_train,
    y_train=y_train_series,
    x_eval=x_eval,
    ref_predictions=ref_predictions,
    model_type=model_type,
    best_params=best_params,
    n_rows=n_rows_to_test,
    threshold=threshold_convergence,
    metric="MeanRelativeError"
)

# Output
print(f"-- Convergence test result ---")
print(f"Test outcome: {convergence_result.passed}")
print(f"Tested subsets (k): {convergence_result.n_rows_tested}")
print(f"Captured errors: {convergence_result.errors}")
print(f"Threshold: {convergence_result.threshold}")
print(f"Minimum k value for convergence: {convergence_result.min_k}")

## Numerical Error Test

In [ ]:
from vv4ml.verification.numerical_error import run_numerical_error_test

# Use the minimum k found in the previous convergence test as the row size. Otherwise, fallback to 100
convergence_k = convergence_result.min_k if convergence_result.min_k else 100

numerical_error_result = run_numerical_error_test(
    x_train=x_train,
    y_train=y_train_df,
    x_eval=x_eval,
    model_type=model_type,
    best_params=best_params,
    n_repetitions=20,
    convergence_rows=convergence_k,
    threshold=0.5,
    random_state=42
)

# Output
print(f"-- Numerical error result ---")
print(f"Test outcome: {numerical_error_result.passed}")
print(f"Maximum standard deviation found: {numerical_error_result.max_variance}")
print(f"Maximum threshold: {numerical_error_result.threshold}")
print(f"Number of ran repetitions: {numerical_error_result.repetitions}")

## Smoothness test

In [ ]:
from vv4ml.verification.smoothness import run_smoothness_test

# Compute real dataset evaluation ranges
input_ranges = [(float(x_eval[col].min()), float(x_eval[col].max())) for col in x_eval.columns]

smoothness_result = run_smoothness_test(
    model=ref_model,
    x_eval=x_eval,
    input_ranges=input_ranges,
    sample_fraction=0.2,                # Test 20% of evaluation points
    perturbation_scaled_magnitude=1e-3, # Delta perturbation
    threshold=2.5,                      # Maximum threshold of the relative error ratio
    random_state=42
)

# Output

print(f"-- Smoothness result ---")
print(f"Test outcome: {smoothness_result.passed}")
print(f"Maximum error ratio: {smoothness_result.max_error_ratio}")
print(f"Threshold: {smoothness_result.threshold}")

## Setup, loading and use of configuration file

Instead of configuring each test manually in your code, `vv4ml` allows you to consolidate all parameters, thresholds, data specs, and model choices into a single declarative configuration file.

Since the configuration file `test_config_file.yaml` is already available locally, we can directly initialize the centralized suite orchestrator to parse the YAML instructions and run all defined verification criteria in a single operational step.

In [ ]:
from vv4ml.config import load_config

conf = load_config("test_config_file.yaml")

# Print some information
print(f"-- Config ---")
print(f"Dataset name: {conf.dataset.name}")
print(f"Train fraction (in train/test split): {conf.model_development.train_test_split.train_fraction}")
print(f"Numerical error threshold: {conf.gate_thresholds.verification.numerical_error[0].value}")

Now, let us set up a simple data loading from the YAML configuration file. We will extract all the necessary model hyperparameters, test thresholds, and specific validation criteria.

In [ ]:
# 1. Model Configuration
conf_model_type = conf.model_development.model_list[0].type
conf_best_params = conf.model_development.model_list[0].hyperparameters

# 2. Convergence Test Parameters
conv_config = conf.test_configurations.verification.convergence
n_rows_to_test = conv_config.n_rows
threshold_convergence = conf.gate_thresholds.verification.convergence[0].value
convergence_metric = conf.gate_thresholds.verification.convergence[0].metric

# 3. Numerical Error Test Parameters
num_config = conf.test_configurations.verification.numerical_error
n_repetitions = num_config.sets
threshold_numerical_error = conf.gate_thresholds.verification.numerical_error[0].value

# 4. Smoothness Test Parameters
smooth_config = conf.test_configurations.verification.smoothness
sample_fraction = smooth_config.sample_fraction
perturbation_scaled_magnitude = smooth_config.perturbation_scaled_magnitude
threshold_smoothness = conf.gate_thresholds.verification.smoothness[0].value

## Convergence Test

In [ ]:
from vv4ml.verification import run_convergence_test

convergence_result = run_convergence_test(
    x_train=x_train,
    y_train=y_train_series,
    x_eval=x_eval,
    ref_predictions=ref_predictions,
    model_type=conf_model_type,
    best_params=conf_best_params,
    n_rows=n_rows_to_test,
    threshold=threshold_convergence,
    metric=convergence_metric
)

# Output
print(f"-- Convergence test result ---")
print(f"Test outcome: {convergence_result.passed}")
print(f"Tested subsets (k): {convergence_result.n_rows_tested}")
print(f"Captured errors: {convergence_result.errors}")
print(f"Threshold: {convergence_result.threshold}")
print(f"Minimum k value for convergence: {convergence_result.min_k}")

## Numerical Error Test

In [ ]:
from vv4ml.verification.numerical_error import run_numerical_error_test

# Use the minimum k found in the previous convergenceperturbation_scaled_magnitude test as the row size. Otherwise, fallback to 100
convergence_k = convergence_result.min_k if convergence_result.min_k else 100

numerical_error_result = run_numerical_error_test(
    x_train=x_train,
    y_train=y_train_df,
    x_eval=x_eval,
    model_type=conf_model_type,
    best_params=conf_best_params,
    n_repetitions=n_repetitions,
    convergence_rows=convergence_k,
    threshold=threshold_numerical_error,
    random_state=42
)

# Output
print(f"-- Numerical error result ---")
print(f"Test outcome: {numerical_error_result.passed}")
print(f"Maximum standard deviation found: {numerical_error_result.max_variance}")
print(f"Maximum threshold: {numerical_error_result.threshold}")
print(f"Number of ran repetitions: {numerical_error_result.repetitions}")

## Smoothness Test

In [ ]:
from vv4ml.verification.smoothness import run_smoothness_test

# Compute real dataset evaluation ranges
input_ranges = [(float(x_eval[col].min()), float(x_eval[col].max())) for col in x_eval.columns]

smoothness_result = run_smoothness_test(
    model=ref_model,
    x_eval=x_eval,
    input_ranges=input_ranges,
    sample_fraction=sample_fraction,                                # Test 20% of evaluation points
    perturbation_scaled_magnitude=perturbation_scaled_magnitude,    # Delta perturbation
    threshold=threshold_smoothness,                            # Maximum threshold of the relative error ratio
    random_state=42
)

# Output

print(f"-- Smoothness result ---")
print(f"Test outcome: {smoothness_result.passed}")
print(f"Maximum error ratio: {smoothness_result.max_error_ratio}")
print(f"Threshold: {smoothness_result.threshold}")